In [0]:
from pyspark.sql import functions as f
from deltalake import DeltaTable

# Identifing the location path of all files in s3 Bucket

In [0]:
%fs ls s3://orders-ecommerce-de/raw

path,name,size,modificationTime
s3://orders-ecommerce-de/raw/customers.csv,customers.csv,245761,1782049384000
s3://orders-ecommerce-de/raw/products.csv,products.csv,83070,1782049384000
s3://orders-ecommerce-de/raw/sales_reps.csv,sales_reps.csv,20062,1782049384000
s3://orders-ecommerce-de/raw/transactions.csv,transactions.csv,625057,1782049384000


# Importing Schemas

In [0]:
%run /Workspace/Users/kharitejareddy997@gmail.com/setup/utlities

In [0]:
df = spark.read.csv('s3://orders-ecommerce-de/raw/sales_reps.csv', header=True, inferSchema=True)


# Writing the data into bronze layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('ecommerce.bronze.sales_reps')

In [0]:
df_silver = spark.read.table('ecommerce.bronze.sales_reps')

# Performing Transformations

In [0]:
# rounding of all the commission_rate values to 2 decimal places
df_silver = df_silver.withColumn('commission_rate', f.abs(f.round(f.col("commission_rate") * 100, 2)))

In [0]:
# defining median value for commission_rate
median_val = df_silver.select("commission_rate").agg(f.median('commission_rate')).collect()[0][0]

In [0]:
# Imputing the median value for the rows with commission_rate is > than 100
df_silver = df_silver.withColumn('commission_rate',
                     f.when(f.col("commission_rate") > 100, median_val)
                     .otherwise(f.col("commission_rate")
                     )
                     )

In [0]:
display(df_silver)

sales_rep_id,rep_name,hire_date,department,territory,commission_rate,email
REP_001,Jane Smith,2022-02-06,SMB,Asia-Pacific,11.42,jane.smith@company.com
REP_002,David Brown,2020-06-07,Direct Sales,Latin America,5.84,david.brown@company.com
REP_003,Emily Williams,2022-08-05,Enterprise,Asia-Pacific,6.62,emily.williams@company.com
REP_004,Jane Rodriguez,2022-08-10,SMB,North America,13.99,jane.rodriguez@company.com
REP_005,Jane Martinez,2020-08-30,SMB,Latin America,11.06,jane.martinez@company.com
REP_006,Robert Smith,2022-06-04,Direct Sales,North America,5.09,robert.smith@company.com
REP_007,John Johnson,2020-06-15,Enterprise,Latin America,10.0,john.johnson@company.com
REP_008,Emily Brown,2022-09-26,Enterprise,Latin America,11.64,emily.brown@company.com
REP_009,William Martinez,2022-03-30,SMB,Europe,5.05,william.martinez@company.com
REP_010,John Rodriguez,2022-04-10,Mid-Market,North America,6.61,john.rodriguez@company.com


In [0]:
df_silver.write.mode('overwrite').option('mergeschema', True).format('delta').saveAsTable('ecommerce.silver.sales_reps')

In [0]:
dim_sales = spark.sql("""
                      select DISTINCT sales_rep_id,  rep_name, department, territory, email, hire_date
                      from ecommerce.silver.sales_reps
                      """)

In [0]:
%sql
drop table ecommerce.gold.dim_sales

In [0]:
dim_sales.write.mode('overwrite').format('delta').saveAsTable('ecommerce.gold.dim_sales')